<a href="https://colab.research.google.com/github/kartixharma/AIML-Experiments/blob/main/LangChain_%26_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community langchainhub chromadb langchain-openai

In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('openaikey')
os.environ['GOOGLE_API_KEY'] = userdata.get('googleapikey')

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(web_path='https://kartixharma-portfolio.vercel.app/')
docs = loader.load()
print(docs)

[Document(metadata={'source': 'https://kartixharma-portfolio.vercel.app/', 'title': 'Kartik Sharma — Software Engineer Portfolio', 'description': 'Modern 2026 portfolio showcasing full-stack, mobile, and AI-driven product engineering work.', 'language': 'en'}, page_content="Kartik Sharma — Software Engineer Portfolio0Loading experienceAvailable for workexperienceprojectsskillscontactBengaluru, India · Full-Stack & MobileKartikSharmaIengineerfast,scalabledigitalproducts—withcinematicmotionandproduct-gradereliability.FromAndroidandReactNativetoNext.jsandFastAPI—Ibuildsystemsthatshipcleanly,performunderscale,anddelivermeasurableuserimpact.View ProjectsResume ↗0Internships0+Production Apps0Graduation0CGPAScrollJavaKotlinCTypeScriptJavaScriptHTMLCSSReact NativePHP LaravelReactNode.jsNext.jsFastAPIAndroid StudioVS CodeGitGitHubFirebaseViteCI/CDMobile + WebProduct EngineeringJavaKotlinCTypeScriptJavaScriptHTMLCSSReact NativePHP LaravelReactNode.jsNext.jsFastAPIAndroid StudioVS CodeGitGitHubFi

In [ ]:
pip install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

In [ ]:
print(len(splits))

5


In [ ]:
pip install -U langchain_openai

In [ ]:
pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.6 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [ ]:
print(vectorstore._collection.count())

5


In [ ]:
print(vectorstore._collection.get())

{'ids': ['a360c285-ec85-482b-939d-a2196c31060e', 'fb18463d-5657-448d-88d4-546d1643cac8', 'b614281b-4447-4264-9e52-6ca53ebef956', '6eb55e44-8466-41e0-8e67-ed9efe314910', 'af1e0eae-7f80-4151-bcea-a5b09b60340d', '1746ec8f-5313-4cee-a1b0-f308e9ae1def', 'ea0d4bc0-ef73-48a9-b3f4-9d2372e381ad', '711b487f-0f29-4b2f-b3a5-4db7575b7d83', 'd4048895-8b26-48c3-bd89-0e8ed445ea3f', 'ca17c86c-41ac-45f2-a374-7bd2edf3d41a', 'c45343b5-8e8d-45a1-bd0e-a7ea682ff807', '9c6252f6-4be4-4b11-8f12-12dffacc05c8', '01c4139d-733e-47cb-8593-95a95d59abac', 'eab3bf79-b53d-43de-9b86-5f62220934b2', '112b74d1-34b5-4cf4-8956-3f3d1c0e32b3', '0ce5830a-c25f-4299-aca0-025582c2469f', 'd73b9190-4cbe-4ff1-bc41-d62a7381fea8', '84bd0852-5678-40eb-9efe-20816d715735', '2ddaf91c-993a-435f-ac29-62a4a9afa6fc', '6ac7e56a-6732-406b-90f3-3a3fb738bf8f', 'ceb01c42-2d2c-4453-a017-cb5aba529666', '3e70bacd-3e39-45c2-996c-7c8f1f9cda4a', '03e47207-fa27-4542-b7a2-6dca8efd7cb5', 'e325a05b-e11f-458d-9976-afeb98b531df', '5374f4e8-0686-4883-ad1d-09c3e7

In [ ]:
vectorstore._collection.delete(ids=vectorstore._collection.get()['ids'])
print(vectorstore._collection.count())

0


In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
llm = ChatOpenAI()

In [ ]:
def format_docs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | model | StrOutputParser())

In [ ]:
rag_chain.invoke("is this candidate fit for full time software developement ?, tell in short")

'Yes, highly skilled and experienced through multiple internships and projects. However, they are currently a student graduating in 2026, meaning they are not available for *immediate* full-time roles.'

In [ ]:
def print_prompt(prompt):
  print("Prompt - ", prompt)
  return prompt

In [ ]:
from langchain_core.runnables import RunnableLambda

rag_chain_with_print = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
             | prompt
             | RunnableLambda(print_prompt)
             | model
             | StrOutputParser())

In [ ]:
rag_chain_with_print.invoke("What is candidate's name")

Prompt -  messages=[HumanMessage(content="\nUse the following context to answer the question.\n\nContext:\n& RuntimesReact NativePHP LaravelReactNode.jsNext.jsFastAPIDeveloper ToolsAndroid StudioVS CodeGitGitHubFirebaseViteDatabasesMongoDBSQLSoft SkillsCreative ThinkingTime ManagementAdaptabilityProblem SolvingEducationDayananda Sagar Academy of Technology and ManagementB.E in Computer Science · 8th SemesterCGPA: 8.1 · 2026 · BengaluruVishal Bharti Public SchoolCBSE Class XIIPercentage: 86% · 2022 · New DelhiCertificationsOracle · AI Foundations AssociateCoding Ninjas · Introduction to JavaCoding Ninjas · Data Structures in JavaUdemy · Jetpack Compose Crash Course for Android with KotlinActivitiesML Learning Hackathon · BMSCETech Tonic Hackathon · Department of CSE, DSATMVolunteeringZenherTechnical Lead (Volunteer)Jun 2025 – Dec 2025 · Remote · IndiaLed development of a menstrual health and period tracking product using React Native, React, and Firebase.Coordinated design, testing, and

'Kartik Sharma'